In [1]:
%matplotlib inline

import anacal
import numpy as np
import matplotlib.pylab as plt

from lsst.skymap.ringsSkyMap import (
    RingsSkyMap, RingsSkyMapConfig
)
from xlens.simulator.catalog import (
    CatalogShearTask,
    CatalogShearTaskConfig,
)
from xlens.simulator.sim import (
    MultibandSimConfig, MultibandSimTask
)
from xlens.processor.measure_coadds import (
    MeasureCoaddsPipeConfig,
    MeasureCoaddsPipe,
)
from xlens.catalog.utils import multiband_shapelets_linear2ell
from xlens.utils.handle import make_exposure_handles

ImportError: /lib64/libc.so.6: version `GLIBC_2.32' not found (required by /hildafs/projects/phy200017p/pladuca/miniforge3/envs/xlens/share/eups/Linux64/geom/g90f42f885a+6054cc57f1/python/lsst/geom/_geom.so)

In [ ]:
config = RingsSkyMapConfig()
config.patchInnerDimensions = [801, 801]
config.tractOverlap = 0.0
config.patchBorder = 0
config.numRings = 7000
config.pixelScale = 0.2
config.projection = "TAN"
skymap = RingsSkyMap(config=config)

config = CatalogShearTaskConfig()
config.extend_ratio = 0.92
config.kappa_value = 0.0000
config.test_value = 0.02
config.layout = "grid"
config.test_target = "g2"
cattask = CatalogShearTask(config=config)

# Multiband Simulation (r, i, z) with 90-degree rotation pairs

Simulate galaxy images in three bands with two rotation sets
(``rotId=0`` and ``rotId=1``, i.e. 0 and 90 degrees), run
``MeasureCoaddsPipe`` with ``return_only_linear_modes=True`` on each,
then combine the catalogs for shear estimation.

In [39]:
# physical bands (butler dimension) for the sim + exposure handles
bands = ["r", "i", "z"]
survey = "lsst"  # MeasureCoaddsPipe default; prefixes output columns as lsst_<band>_...

# Simulate two rotation sets: rotId=0 (0 deg) and rotId=1 (90 deg)
outcomes_list = {}
truths = {}
for rot_id in [0, 1]:
    # Generate truth catalog with matching rotId
    cat_config = CatalogShearTaskConfig()
    cat_config.extend_ratio = 0.92
    cat_config.kappa_value = 0.0000
    cat_config.test_value = 0.02
    cat_config.layout = "grid"
    cat_config.test_target = "g2"
    cat_config.rotId = rot_id
    cat_task = CatalogShearTask(config=cat_config)
    truths[rot_id] = cat_task.run(
        tract_info=skymap[0],
        seed=0,
    ).truthCatalog

    simtask_config = MultibandSimConfig()
    simtask_config.survey_name = "lsst"
    simtask_config.rotId = rot_id
    simtask = MultibandSimTask(config=simtask_config)

    outcomes = {}
    for band in bands:
        outcomes[band] = simtask.run(
            tract_info=skymap[0],
            patch_id=0,
            band=band,
            seed=0,
            truthCatalog=truths[rot_id],
        )
        print(f"rotId={rot_id}, Band {band}: simulated")
    outcomes_list[rot_id] = outcomes

rotId=0, Band r: simulated
rotId=0, Band i: simulated
rotId=0, Band z: simulated
rotId=1, Band r: simulated
rotId=1, Band i: simulated
rotId=1, Band z: simulated


In [ ]:
# Build per-rotation detection catalogs from truth positions.
wcs = skymap[0].getWcs()
pixel_scale = 0.2

detections = {}
for rot_id in [0, 1]:
    px, py = wcs.skyToPixelArray(
        truths[rot_id]["ra"], truths[rot_id]["dec"], degrees=True
    )
    detections[rot_id] = anacal.table.make_catalog_empty(
        px * pixel_scale, py * pixel_scale
    )
    print(f"rotId={rot_id}: {len(detections[rot_id])} detections")

In [ ]:
# Configure the measurement pipeline with linear modes.
# Disable noise bias correction since the simulation is noiseless.
pipe_config = MeasureCoaddsPipeConfig()
pipe_config.anacal.force_size = True
pipe_config.anacal.force_center = True
pipe_config.anacal.do_noise_bias_correction = False
pipe_config.fpfs.do_compute_detect_weight = False
pipe_config.fpfs.sigma_shapelets1 = 0.52
pipe_config.fpfs.do_noise_bias_correction = False
pipe_config.fpfs.return_only_linear_modes = True

pipe = MeasureCoaddsPipe(config=pipe_config)

# Measure both rotation sets
cats = {}
for rot_id in [0, 1]:
    exposure_handles_dict = make_exposure_handles(
        {b: outcomes_list[rot_id][b].simExposure for b in bands},
        tract=0, patch=0,
    )
    cats[rot_id] = pipe.run(
        exposure_handles_dict=exposure_handles_dict,
        corr_array=None,
        skyMap=skymap,
        tract=0,
        patch=0,
        detection=detections[rot_id],
    ).anacalCatalog
    print(f"rotId={rot_id}: {len(cats[rot_id])} sources measured")

# Combine rotation pairs and estimate shear

Convert linear modes to ellipticities for each rotation set, concatenate
the two catalogs, then estimate shear. The 90-degree rotation cancels
shape noise from intrinsic galaxy ellipticities.

In [42]:
import numpy.lib.recfunctions as rfn

prefix = "fpfs1_"
C0 = 8.4

# Convert linear modes to ellipticities for each rotation set
ell_list = []
for rot_id in [0, 1]:
    ell = multiband_shapelets_linear2ell(
        cats[rot_id], bands=[f"{survey}_{b}" for b in bands], C0=C0, prefix=prefix
    )
    ell_list.append(ell)

# Concatenate the two rotation sets
ell_combined = rfn.stack_arrays(ell_list, usemask=False)

In [46]:
print("Shear Response Matrix")
print(np.mean(ell_combined[f"{prefix}de1_dg1"]), np.mean(ell_combined[f"{prefix}de1_dg2"]))
print(np.mean(ell_combined[f"{prefix}de1_dg2"]), np.mean(ell_combined[f"{prefix}de2_dg2"]))

Shear Response Matrix
0.16230458506953882 0.000407478506575453
0.000407478506575453 0.16380159047472428


# Shear estimation

$$\hat{g}_i = \frac{\langle e_i \rangle}{\langle \partial e_i / \partial g_i \rangle}$$

In [43]:
g1_mean = (
    np.sum(ell_combined[f"{prefix}e1"])
    / np.sum(ell_combined[f"{prefix}de1_dg1"])
)
g2_mean = (
    np.sum(ell_combined[f"{prefix}e2"])
    / np.sum(ell_combined[f"{prefix}de2_dg2"])
)

print(f"g1 = {g1_mean:.6f} (expected ~0)")
print(f"g2 = {g2_mean:.6f} (expected ~-0.02)")

g1 = -0.000050 (expected ~0)
g2 = -0.020004 (expected ~-0.02)


In [44]:
assert np.abs(g1_mean) < 1e-3, f"g1 bias too large: {g1_mean}"
assert np.abs((g2_mean + 0.02) / 0.02) < 0.01, (
    f"g2 fractional bias too large: {(g2_mean + 0.02) / 0.02}"
)
print("Shear recovery test passed!")

Shear recovery test passed!
